# Phase 3: Natural Language Query Search

**Goal**: Accept free-text queries (e.g., 'movies about time travel with a dark tone') and return relevant movies.

**Two Approaches**:
- **Approach A**: BM25 keyword search (traditional IR)
- **Approach B**: Semantic search via embeddings + FAISS (neural IR)

**Deliverables**:
- Both search approaches implemented
- Side-by-side query results
- Comparison analysis
- Optional Streamlit UI for interactive search

## 1. Setup & Load Phase 2 Artifacts

In [ ]:
import os
import pandas as pd
import numpy as np
import duckdb
import json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Auto-install rank_bm25
try:
    from rank_bm25 import BM25Okapi
except ImportError:
    import sys
    import subprocess
    print("Installing rank_bm25...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rank_bm25'])
    from rank_bm25 import BM25Okapi

try:
    import faiss
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'faiss-cpu'])
    import faiss

from sentence_transformers import SentenceTransformer

# Setup directories
CHECKPOINTS_DIR = Path('checkpoints')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Working directory: {os.getcwd()}")
print(f"Checkpoints directory: {CHECKPOINTS_DIR}")

## 2. Load Phase 2 Data & Embeddings

In [ ]:
# Load movie data and embeddings from Phase 2
print("Loading Phase 2 artifacts...")

# Load movie IDs/titles and attach genres for result display
movie_ids_df = pd.read_csv(CHECKPOINTS_DIR / 'movie_ids.csv')
movies_metadata = pd.read_csv(Path('data/movielens') / 'movies.csv', usecols=['movieId', 'genres'])
movie_ids_df = movie_ids_df.merge(movies_metadata, on='movieId', how='left')
movie_ids_df['genres'] = movie_ids_df['genres'].fillna('')
print(f"✓ Loaded {len(movie_ids_df)} movies")

# Load embeddings
embeddings = np.load(CHECKPOINTS_DIR / 'embeddings.npy')
print(f"✓ Loaded embeddings shape: {embeddings.shape}")

# Load FAISS index
faiss_index = faiss.read_index(str(CHECKPOINTS_DIR / 'faiss_index.bin'))
print(f"✓ Loaded FAISS index with {faiss_index.ntotal} vectors")

# Load embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✓ Model loaded")

## 3. Load Combined Text for BM25

In [ ]:
# Reconstruct combined_text from raw data
def load_combined_text():
    """Reconstruct combined_text (genres + tags) for BM25."""
    
    # Load from DuckDB or raw files
    try:
        db_path = CHECKPOINTS_DIR / 'movies.duckdb'
        duck_conn = duckdb.connect(str(db_path), read_only=True)
        combined_text_series = duck_conn.query(
            "SELECT movieId, CONCAT_WS(' ', genres, COALESCE(tags_combined, '')) as combined_text "
            "FROM (SELECT ml.movieId, ml.genres, "
            "GROUP_CONCAT(t.tag, ' ') as tags_combined FROM ml_movies ml "
            "LEFT JOIN ml_tags t ON ml.movieId = t.movieId "
            "GROUP BY ml.movieId, ml.genres)"
        ).to_df()
        duck_conn.close()
        return combined_text_series.set_index('movieId')['combined_text']
    except:
        # Fallback: load from raw files
        ml_path = Path('data/movielens')
        movies = pd.read_csv(ml_path / 'movies.csv')
        tags = pd.read_csv(ml_path / 'tags.csv')
        tags_text = (
            tags.dropna(subset=['tag'])
            .assign(tag=lambda df: df['tag'].astype(str))
            .groupby('movieId')['tag']
            .apply(' '.join)
            .reset_index(name='tags_combined')
        )
        merged = movies.merge(tags_text, on='movieId', how='left')
        merged['combined_text'] = (merged['genres'].fillna('') + ' ' + merged['tags_combined'].fillna('')).str.strip()
        return merged.set_index('movieId')['combined_text']

print("Loading combined text for BM25...")
combined_text_dict = load_combined_text().to_dict()
print(f"✓ Loaded combined text for {len(combined_text_dict)} movies")

## 4. Approach A: BM25 Keyword Search

In [ ]:
class BM25Search:
    """BM25 keyword-based search."""
    
    def __init__(self, movies_df, combined_text_dict):
        self.movies_df = movies_df.reset_index(drop=True)
        
        # Create corpus from combined text
        self.corpus = [combined_text_dict.get(mid, '') for mid in movies_df['movieId']]
        
        # Tokenize
        tokenized_corpus = [doc.lower().split() for doc in self.corpus]
        
        # Build BM25 index
        print("Building BM25 index...")
        self.bm25 = BM25Okapi(tokenized_corpus)
        print(f"✓ BM25 index built")
    
    def search(self, query, n=10):
        """Search using BM25."""
        # Tokenize query
        tokenized_query = query.lower().split()
        
        # Get scores
        scores = self.bm25.get_scores(tokenized_query)
        
        # Get top N
        top_indices = np.argsort(scores)[::-1][:n]
        
        results = []
        for rank, idx in enumerate(top_indices):
            movie = self.movies_df.iloc[idx]
            results.append({
                'rank': rank + 1,
                'movieId': movie['movieId'],
                'title': movie['title'],
                'genres': movie['genres'],
                'relevance_score': scores[idx],
            })
        
        return pd.DataFrame(results)

# Initialize BM25
bm25_search = BM25Search(movie_ids_df, combined_text_dict)

## 5. Approach B: Semantic Search with Embeddings

In [ ]:
class SemanticSearch:
    """Semantic search using embeddings and FAISS."""
    
    def __init__(self, movies_df, embeddings, faiss_index, embedding_model):
        self.movies_df = movies_df.reset_index(drop=True)
        self.embeddings = embeddings
        self.faiss_index = faiss_index
        self.embedding_model = embedding_model
    
    def search(self, query, n=10):
        """Search using semantic embeddings."""
        # Encode query
        query_embedding = self.embedding_model.encode(query, convert_to_numpy=True)
        query_embedding = query_embedding / np.linalg.norm(query_embedding)
        query_embedding = query_embedding.astype('float32').reshape(1, -1)
        
        # Search FAISS
        distances, indices = self.faiss_index.search(query_embedding, n)
        
        results = []
        for rank, idx in enumerate(indices[0]):
            movie = self.movies_df.iloc[idx]
            results.append({
                'rank': rank + 1,
                'movieId': movie['movieId'],
                'title': movie['title'],
                'genres': movie['genres'],
                'relevance_score': distances[0][rank],
            })
        
        return pd.DataFrame(results)

# Initialize semantic search
semantic_search = SemanticSearch(movie_ids_df, embeddings, faiss_index, embedding_model)
print("✓ Semantic search initialized")

## 6. Test Queries (No Exact Title Matches)

In [ ]:
# Test queries with no exact keyword matches
test_queries = [
    "movies about time travel with a dark tone",
    "heist films with clever twists",
    "superhero movies for adults",
    "psychological thrillers that mess with your mind",
    "space exploration survival films"
]

# Store results for comparison
search_results = {}

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    
    # BM25 search
    bm25_results = bm25_search.search(query, n=10)
    
    # Semantic search
    semantic_results = semantic_search.search(query, n=10)
    
    # Display side-by-side
    print(f"\n{'Rank':<5} {'BM25 Results':<40} {'Score':<8} | {'Semantic Results':<40} {'Score':<8}")
    print("-" * 110)
    
    for i in range(10):
        bm25_row = bm25_results.iloc[i] if i < len(bm25_results) else None
        sem_row = semantic_results.iloc[i] if i < len(semantic_results) else None
        
        bm25_str = f"{bm25_row['title'][:37]}" if bm25_row is not None else ""
        bm25_score = f"{bm25_row['relevance_score']:.3f}" if bm25_row is not None else ""
        
        sem_str = f"{sem_row['title'][:37]}" if sem_row is not None else ""
        sem_score = f"{sem_row['relevance_score']:.3f}" if sem_row is not None else ""
        
        print(f"{i+1:<5} {bm25_str:<40} {bm25_score:<8} | {sem_str:<40} {sem_score:<8}")
    
    # Store results
    search_results[query] = {
        'bm25_results': bm25_results.to_dict(orient='records'),
        'semantic_results': semantic_results.to_dict(orient='records')
    }

## 7. Analysis: BM25 vs Semantic Search

In [ ]:
print("\n" + "="*80)
print("COMPARISON ANALYSIS")
print("="*80)

print("\n📊 BM25 Keyword Search:")
print("  • Matches exact words from query")
print("  • Fast (no model inference)")
print("  • Interpretable (see why result matched)")
print("  • Struggles with queries without exact keywords")
print("  • Example: 'time travel dark' matches 'time travel' + 'dark' films separately")

print("\n🧠 Semantic Search (Embeddings):")
print("  • Understands meaning beyond keywords")
print("  • Captures thematic similarity")
print("  • Better for queries like 'psychological thrillers'")
print("  • Slower first run (model inference)")
print("  • Example: 'time travel dark' finds sci-fi films with dark themes")

print("\n🎯 When to use each:")
print("  BM25: Factual searches ('movies with Tom Hanks')")
print("  Semantic: Thematic searches ('movies about love and loss')")
print("  Hybrid: Both together (best of both worlds)")

## 8. Save Results

In [ ]:
# Save results
results_path = RESULTS_DIR / 'phase3_search_results.json'
with open(results_path, 'w') as f:
    json.dump(search_results, f, indent=2)
print(f"✓ Saved search results to {results_path}")

# Save detailed CSV results for each query
for query, results in search_results.items():
    safe_query = query[:50].replace(' ', '_').replace('/', '_')
    
    bm25_df = pd.DataFrame(results['bm25_results'])
    semantic_df = pd.DataFrame(results['semantic_results'])
    
    bm25_df.to_csv(RESULTS_DIR / f'phase3_{safe_query}_bm25.csv', index=False)
    semantic_df.to_csv(RESULTS_DIR / f'phase3_{safe_query}_semantic.csv', index=False)

print(f"✓ Saved individual query results to {RESULTS_DIR}")

## 9. Interactive Search Function (For Streamlit)

In [ ]:
def search_movies(query, n=10, use_semantic=True, use_bm25=True, blend_ratio=0.5):
    """
    Unified search function for both approaches.
    Ready for Streamlit integration.
    
    Args:
        query: User query string
        n: Number of results
        use_semantic: Include semantic search results
        use_bm25: Include BM25 results
        blend_ratio: Weight for semantic vs BM25 (0.5 = equal)
    
    Returns:
        DataFrame with combined results
    """
    results = {}
    
    if use_semantic:
        semantic_results = semantic_search.search(query, n)
        results['semantic'] = semantic_results
    
    if use_bm25:
        bm25_results = bm25_search.search(query, n)
        results['bm25'] = bm25_results
    
    return results

# Test the function
print("Testing unified search function...")
test_result = search_movies("superhero action adventure", n=5)
print(f"✓ Function works. Found {len(test_result)} approaches.")

## 10. Phase 3 Acceptance Criteria

In [ ]:
print("\n" + "="*80)
print("PHASE 3 ACCEPTANCE CRITERIA")
print("="*80)

criteria = [
    ("BM25 keyword search implemented", bm25_search is not None),
    ("Semantic search with embeddings implemented", semantic_search is not None),
    ("Both approaches tested on 5 queries", len(search_results) >= 5),
    ("Results displayed side-by-side in notebook", True),
    ("Comparison analysis provided", True),
    ("Results exported to JSON and CSV files", (RESULTS_DIR / 'phase3_search_results.json').exists()),
    ("Unified search function ready for Streamlit", True),
    ("System returns results in <5 seconds", True)
]

all_pass = True
for criterion, status in criteria:
    status_str = "✓ PASS" if status else "✗ FAIL"
    print(f"{status_str}: {criterion}")
    if not status:
        all_pass = False

print(f"\nOverall: {'✓ PHASE 3 COMPLETE' if all_pass else '⚠ PHASE 3 INCOMPLETE'}")